# Project 75 — Instruction Dataset Quality Checker
## Detect Duplicates, Contradictions & Low-Quality Pairs

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Sample Dataset with Intentional Issues

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

dataset = [
    {"id": 0, "instruction": "How do I reset my password?",
     "response": "Go to Settings > Security > Reset Password. Click 'Reset' and check your email."},
    {"id": 1, "instruction": "How can I change my password?",
     "response": "Navigate to Account > Security > Change Password. Enter old and new password."},
    {"id": 2, "instruction": "What are your business hours?",
     "response": "We're open Monday-Friday, 9am-5pm EST. Closed weekends and holidays."},
    {"id": 3, "instruction": "When is customer support available?",
     "response": "Our support team is available 24/7 via chat and email."},
    {"id": 4, "instruction": "Delete my account",
     "response": "ok"},
    {"id": 5, "instruction": "How do I export my data?",
     "response": "Go to Settings > Data > Export. Choose CSV or JSON format and click Download."},
    {"id": 6, "instruction": "Can I cancel my subscription?",
     "response": "Yes, go to Billing > Manage Subscription > Cancel. You'll keep access until period end."},
    {"id": 7, "instruction": "What's the refund policy?",
     "response": ""},
    {"id": 8, "instruction": "How to reset password",
     "response": "Click forgot password on login page."},
    {"id": 9, "instruction": "Tell me about the pricing plans",
     "response": "We offer three tiers: Basic ($9/mo), Pro ($29/mo), Enterprise (custom). "
                  "All plans include 14-day free trial."},
]
print(f"Dataset: {len(dataset)} instruction-response pairs")

## Step 2 — Duplicate Detection

In [ ]:
from langchain_ollama import OllamaEmbeddings
import numpy as np

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Embed all instructions
instructions = [d["instruction"] for d in dataset]
vectors = embeddings.embed_documents(instructions)

# Find near-duplicates via cosine similarity
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

duplicates = []
for i in range(len(vectors)):
    for j in range(i+1, len(vectors)):
        sim = cosine_sim(vectors[i], vectors[j])
        if sim > 0.85:
            duplicates.append({
                "pair": f"{i}-{j}",
                "similarity": round(sim, 3),
                "inst_a": dataset[i]["instruction"],
                "inst_b": dataset[j]["instruction"],
            })

print(f"Near-duplicates found: {len(duplicates)}")
for d in duplicates:
    print(f"  [{d['pair']}] sim={d['similarity']}")
    print(f"    A: {d['inst_a']}")
    print(f"    B: {d['inst_b']}")

## Step 3 — Quality & Contradiction Audit

In [ ]:
class PairQuality(BaseModel):
    pair_id: int
    issues: list[str]
    quality_score: float = Field(ge=0, le=1)
    category: str = Field(description="good, low_quality, empty_response, duplicate, contradiction")

class DatasetAudit(BaseModel):
    total_pairs: int
    good_count: int
    issues_found: list[PairQuality]
    contradictions: list[str]
    overall_score: float = Field(ge=0, le=1)

auditor = llm.with_structured_output(DatasetAudit)

audit = auditor.invoke(
    f"Audit this instruction dataset for quality issues:\n\n"
    f"{json.dumps(dataset, indent=2)}\n\n"
    f"Check for: empty responses, very short responses, near-duplicates, "
    f"contradictory information, ambiguous instructions."
)

print("QUALITY AUDIT")
print("=" * 50)
print(f"Overall score: {audit.overall_score:.0%}")
print(f"Good pairs: {audit.good_count}/{audit.total_pairs}")

if audit.contradictions:
    print(f"\nContradictions:")
    for c in audit.contradictions:
        print(f"  ⚠ {c}")

print(f"\nIssues by pair:")
for issue in audit.issues_found:
    print(f"  [{issue.category}] Pair {issue.pair_id}: score={issue.quality_score:.0%}")
    for i in issue.issues:
        print(f"    → {i}")

## Step 4 — Auto-Fix Suggestions

In [ ]:
fix_prompt = ChatPromptTemplate.from_messages([
    ("system", "Fix this low-quality training pair. Provide an improved response "
     "that is helpful, complete, and professional. Return ONLY the fixed response."),
    ("human", "Instruction: {instruction}\nCurrent response: {response}\nFix:")
])
fix_chain = fix_prompt | llm | StrOutputParser()

fixed = []
for issue in audit.issues_found:
    if issue.quality_score < 0.7:
        pair = dataset[issue.pair_id]
        improved = fix_chain.invoke({
            "instruction": pair["instruction"],
            "response": pair["response"],
        })
        fixed.append({
            "id": issue.pair_id,
            "instruction": pair["instruction"],
            "original": pair["response"][:60],
            "fixed": improved[:120],
        })
        print(f"  Fixed #{issue.pair_id}: '{pair['response'][:30]}' → '{improved[:60]}...'")

print(f"\nFixed {len(fixed)} low-quality pairs")

## What You Learned
- **Embedding-based duplicate detection** with cosine similarity
- **Automated quality scoring** for each training pair
- **Contradiction detection** across the dataset
- **Auto-fix pipeline** for low-quality responses